# Raw-space inverse-mask panels for selected ROIs

This notebook renders one raw-space panel per ROI for the 1050 dataset.
Each figure shows 5 days across columns and z-5 to z+5 across rows, using
the day-0 SAM mask plus the inverse-warped raw-space masks for later days.
The plotting logic lives in `raw_space_triplet_panels.py` so the layout can
be edited later without touching the main processing pipeline.


In [ ]:
from pathlib import Path

import pandas as pd
import tifffile
from IPython.display import Image, display

from analysis_paths import get_dataset_analysis_dir, get_shape_qc_analysis_dir, resolve_dataset_dir
from roi_log_ratio_analysis import build_inverse_warped_mask_lookup, build_raw_image_lookup
from run_registered_roi_pipeline import infer_start_date_from_dataset_dir
from raw_space_triplet_panels import render_raw_space_triplet_panel


In [ ]:
# Dataset and input configuration
dataset_dir = resolve_dataset_dir("1050")
analysis_dir = get_dataset_analysis_dir("1050")
shape_qc_dir = get_shape_qc_analysis_dir("1050")
start_date = infer_start_date_from_dataset_dir(dataset_dir)

day0_mask_name = "mean_image_merge_cp_masks_SAM.tif"
inverse_mask_suffix = "_ROI_mask_SyN_inversed.tif"
preferred_channel = "red"

# These are the selected ROIs for the figure set.
roi_ids = [1382, 1379, 1607]

summary_path = shape_qc_dir / "roi_log_ratio_summary_size_and_shape_filtered.csv"
metrics_path = shape_qc_dir / "roi_log_ratio_metrics_size_and_shape_filtered.csv"

output_dir = analysis_dir / f"raw_space_triplet_roi_panels_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"dataset_dir={dataset_dir}")
print(f"shape_qc_dir={shape_qc_dir}")
print(f"output_dir={output_dir}")
print(f"start_date={start_date}")


In [ ]:
# Load the filtered tables and confirm the requested ROIs survived the regular QC steps.
roi_summary = pd.read_csv(summary_path).set_index("roi_id")
roi_metrics = pd.read_csv(metrics_path)

missing_roi_ids = sorted(set(roi_ids).difference(roi_summary.index.astype(int).tolist()))
if missing_roi_ids:
    raise ValueError(f"Selected ROIs missing from the filtered summary table: {missing_roi_ids}")

present_metrics_ids = set(roi_metrics["roi_id"].astype(int).unique())
missing_in_metrics = sorted(set(roi_ids).difference(present_metrics_ids))
if missing_in_metrics:
    raise ValueError(f"Selected ROIs missing from the filtered metrics table: {missing_in_metrics}")

selected_summary = roi_summary.loc[roi_ids].copy()
selected_summary


In [ ]:
# Load the raw image stacks and the day-specific inverse-warped masks once.
raw_lookup_paths = build_raw_image_lookup(dataset_dir, start_date=start_date)
mask_lookup_paths = build_inverse_warped_mask_lookup(
    image_dir=dataset_dir,
    day0_mask_name=day0_mask_name,
    inverse_mask_suffix=inverse_mask_suffix,
    preferred_channel=preferred_channel,
    start_date=start_date,
)

raw_stack_lookup = {
    (day, channel): tifffile.imread(path).astype(float)
    for (day, channel), path in sorted(raw_lookup_paths.items())
}
mask_stack_lookup = {day: tifffile.imread(path) for day, path in sorted(mask_lookup_paths.items())}

print(f"days={sorted(mask_stack_lookup)}")
print(f"raw_stacks={len(raw_stack_lookup)}")
print(f"mask_stacks={len(mask_stack_lookup)}")


In [ ]:
# Render and save one panel per ROI.
all_metadata_rows = []
for roi_id in roi_ids:
    roi_row = roi_summary.loc[roi_id]
    output_path = output_dir / f"roi_{roi_id}_raw_space_triplet_panel.png"
    metadata_rows = render_raw_space_triplet_panel(
        roi_id=roi_id,
        raw_stack_lookup=raw_stack_lookup,
        mask_stack_lookup=mask_stack_lookup,
        output_path=output_path,
        roi_summary=roi_row,
        start_date=start_date,
        half_window_z=5,
        crop_pad_xy=20,
        min_crop_size_px=48,
    )
    all_metadata_rows.extend(metadata_rows)
    display(Image(filename=str(output_path)))

metadata_path = output_dir / "raw_space_triplet_panel_metadata.csv"
pd.DataFrame(all_metadata_rows).to_csv(metadata_path, index=False)
metadata_path
